In [2]:
!pip install gliner -U

In [ ]:
import pandas as pd
import json
import os
from utils.path_loader import load_paths

In [ ]:
paths = load_paths()
PROJECT_ROOT = paths["project_root"]

PATH_INPUT = PROJECT_ROOT /"data"/"processed"/"skills_filtradas"/"df_skills_final.pkl"  
PASTA_OUTPUT = PROJECT_ROOT/"data"/"processed_GliNER"/"GliNER_dataset"       
NOME_FICHEIRO_SAIDA = "gliner_train_data.json"

def convert_to_gliner_format():
    # 1. Verificar e criar a pasta de output se não existir
    if not os.path.exists(PASTA_OUTPUT):
        os.makedirs(PASTA_OUTPUT)
        print(f"[*] Pasta criada: {PASTA_OUTPUT}")

    caminho_final = os.path.join(PASTA_OUTPUT, NOME_FICHEIRO_SAIDA)

    # 2. Carregar o DataFrame
    try:
        df = pd.read_pickle(PATH_INPUT)
        print(f"[*] Ficheiro {PATH_INPUT} carregado com sucesso.")
    except Exception as e:
        print(f"[!] Erro ao carregar o ficheiro: {e}")
        return

    gliner_dataset = []

    # 3. Processar cada linha do DataFrame
    for index, row in df.iterrows():
        text = row['text']
        raw_annotations = row['annotations']
        
        ner_spans = []
        for start, end, label in raw_annotations:
            # Limpeza: remove 'SKILL: ' e deixa em minúsculas
            clean_label = label.replace("SKILL: ", "").strip().lower()
            
            # Validação técnica dos índices
            if start < end and end <= len(text):
                ner_spans.append([start, end, clean_label])
        
        # O GLiNER pode aceitar 'text' ou 'tokenized_text'. 
        # O formato de texto corrido com spans de caracteres é o mais simples para começar.
        gliner_dataset.append({
            "text": text,
            "ner": ner_spans
        })

    # 4. Guardar o ficheiro JSON final
    try:
        with open(caminho_final, "w", encoding="utf-8") as f:
            json.dump(gliner_dataset, f, ensure_ascii=False, indent=4)
        print(f"\n[SUCESSO]")
        print(f"Total de CVs processados: {len(gliner_dataset)}")
        print(f"Ficheiro guardado em: {caminho_final}")
    except Exception as e:
        print(f"[!] Erro ao guardar o ficheiro JSON: {e}")

if __name__ == "__main__":
    convert_to_gliner_format()

[*] Pasta criada: PROJECT_ROOT/data/processed_GliNER/GliNER_dataset
[!] Erro ao carregar o ficheiro: [Errno 2] No such file or directory: 'PROJECT_ROOT/data/processed/skills_filtradas/df_skills_final.pkl'
